# 002 · A Simple HTTP Server

> **Standard Library › Networking** · Python 3.14

`http.server` contains a small, dependency-free HTTP server. You subclass `BaseHTTPRequestHandler`, implement `do_GET`, `do_POST`, ... and write the response yourself — similar in spirit to .NET's `HttpListener`. It is ideal for test doubles, local tools and learning how HTTP works, but **not** for production.

**In this notebook you will learn:**

- how `BaseHTTPRequestHandler` maps onto ASP.NET Core minimal APIs and `HttpListener`
- reading the request: `path`, `command`, `headers`, and the body via `Content-Length` + `rfile`
- writing the response: `send_response`, `send_header`, `end_headers`, `wfile`
- simple routing for a small JSON API
- `HTTPServer` vs `ThreadingHTTPServer`, running in a background thread, and shutting down
- serving static files and the `python -m http.server` command line

## .NET mental model

| .NET | Python `http.server` |
|---|---|
| `HttpListener` | `HTTPServer` / `ThreadingHTTPServer` |
| `HttpListenerContext` (request + response) | one `BaseHTTPRequestHandler` instance per request |
| `app.MapGet("/todos", ...)` | `do_GET` + your own routing on `self.path` |
| `request.HttpMethod` / `Url` / `Headers` | `self.command` / `self.path` / `self.headers` |
| `request.InputStream` | `self.rfile` (read exactly `Content-Length` bytes) |
| `response.StatusCode = 201` | `self.send_response(201)` |
| `response.Headers.Add(...)` | `self.send_header(name, value)` + `self.end_headers()` |
| `response.OutputStream` | `self.wfile` |
| `Results.NotFound()` | `self.send_error(404)` |
| `StatusCodes.Status201Created` | `http.HTTPStatus.CREATED` |
| `app.UseStaticFiles()` / `dotnet serve` | `SimpleHTTPRequestHandler` / `python -m http.server` |
| Kestrel + ASP.NET Core | a WSGI/ASGI server + framework (third-party) |

⚠️ **Gotcha:** the Python docs state plainly that `http.server` is **not recommended for production** — it only implements basic security checks. Real Python web apps use a framework on top of the **WSGI** (Flask, Django) or **ASGI** (FastAPI, Starlette, Django async) standard, served by a production server such as gunicorn or uvicorn. Those are the ASP.NET Core + Kestrel of the Python world.

## The smallest possible server

```csharp
// C# minimal API
var app = WebApplication.Create();
app.MapGet("/hello", () => "Hello, world!");
app.Run("http://127.0.0.1:5000");
```

With `http.server` there are no route attributes or result helpers: you write status line, headers and body yourself. The server creates a **new handler instance for every request** and calls the `do_<METHOD>` method that matches the HTTP verb.

In [ ]:
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer

class HelloHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        body = f"Hello from {self.path}\n".encode("utf-8")
        self.send_response(200)                                   # status line (+ Server/Date headers)
        self.send_header("Content-Type", "text/plain; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()                                        # blank line: headers done
        self.wfile.write(body)                                    # the body

    def log_message(self, format, *args):                        # default logs every request to stderr
        pass

hello_server = HTTPServer(("127.0.0.1", 0), HelloHandler)        # port 0 → OS picks a free port
hello_port = hello_server.server_address[1]
threading.Thread(target=hello_server.serve_forever, daemon=True).start()
print("listening:", hello_port > 0)

`serve_forever()` blocks, so we run it in a **daemon thread** *(see Standard Library › Concurrency › 001 · Threads)* and talk to it from the notebook with `urllib` *(see Standard Library › Networking › 001 · HTTP Requests with urllib)*:

In [ ]:
from urllib.request import urlopen

with urlopen(f"http://127.0.0.1:{hello_port}/greeting?name=alice", timeout=5) as r:
    print(r.status, r.headers["Content-Type"])
    print(r.read().decode())

⚠️ **Gotcha:** the order is fixed: `send_response` → `send_header` (any number) → `end_headers` → write the body. Forgetting `end_headers()` means the client never sees the response. And send a `Content-Length` — without it the client can only detect the end of the body when the connection closes.

**Tip:** the default `log_message` writes an Apache-style access log line to stderr for every request. Overriding it (as above) keeps notebook output clean; in a tool you'd typically route it to `logging`.

Shut the server down: `shutdown()` stops the `serve_forever` loop (call it from a *different* thread than the one serving), and `server_close()` closes the listening socket — the `HttpListener.Stop()` + `Close()` pair.

In [ ]:
hello_server.shutdown()
hello_server.server_close()
print("hello server stopped")

## A small JSON API with routing

Let's build a to-do API:

| Method | Path | Result |
|---|---|---|
| `GET` | `/todos` | list all |
| `GET` | `/todos/<id>` | one item or 404 |
| `POST` | `/todos` | create from JSON body → 201 + `Location` header |
| `DELETE` | `/todos/<id>` | 204 or 404 |

```csharp
// C# minimal API equivalent
app.MapGet("/todos", () => store.Values);
app.MapGet("/todos/{id:int}", (int id) => store.TryGetValue(id, out var t) ? Results.Ok(t) : Results.NotFound());
app.MapPost("/todos", (Todo todo) => { /* ... */ return Results.Created($"/todos/{id}", todo); });
app.MapDelete("/todos/{id:int}", (int id) => store.Remove(id) ? Results.NoContent() : Results.NotFound());
```

There is no router in `http.server`, but `match` with a pattern on the split path works nicely for small cases *(see Syntax › Control Flow › 002 · The match Statement)*. Shared state needs a lock, because `ThreadingHTTPServer` handles requests on multiple threads.

In [ ]:
import json
from http import HTTPStatus
from http.server import ThreadingHTTPServer
from urllib.parse import urlparse

todos = {1: {"id": 1, "title": "write docs", "done": False}}
next_id = 2
store_lock = threading.Lock()

class TodoHandler(BaseHTTPRequestHandler):
    # ---- helpers -------------------------------------------------------
    def send_json(self, status, payload=None, headers=None):
        body = b"" if payload is None else json.dumps(payload).encode("utf-8")
        self.send_response(status)
        for name, value in (headers or {}).items():
            self.send_header(name, value)
        if payload is not None:
            self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def read_json(self):
        length = int(self.headers.get("Content-Length", 0))
        return json.loads(self.rfile.read(length)) if length else None

    def segments(self):
        return [s for s in urlparse(self.path).path.split("/") if s]

    def log_message(self, format, *args):
        pass

    # ---- routes --------------------------------------------------------
    def do_GET(self):
        match self.segments():
            case ["todos"]:
                with store_lock:
                    self.send_json(HTTPStatus.OK, sorted(todos.values(), key=lambda t: t["id"]))
            case ["todos", todo_id] if todo_id.isdigit():
                with store_lock:
                    todo = todos.get(int(todo_id))
                if todo is None:
                    self.send_json(HTTPStatus.NOT_FOUND, {"error": f"todo {todo_id} not found"})
                else:
                    self.send_json(HTTPStatus.OK, todo)
            case _:
                self.send_error(HTTPStatus.NOT_FOUND, "Unknown route")

    def do_POST(self):
        global next_id
        if self.segments() != ["todos"]:
            return self.send_error(HTTPStatus.NOT_FOUND)
        try:
            data = self.read_json()
            title = data["title"]
        except (json.JSONDecodeError, TypeError, KeyError):
            return self.send_json(HTTPStatus.BAD_REQUEST, {"error": "JSON body with 'title' required"})
        with store_lock:
            todo = {"id": next_id, "title": title, "done": False}
            todos[next_id] = todo
            next_id += 1
        self.send_json(HTTPStatus.CREATED, todo, headers={"Location": f"/todos/{todo['id']}"})

    def do_DELETE(self):
        match self.segments():
            case ["todos", todo_id] if todo_id.isdigit():
                with store_lock:
                    removed = todos.pop(int(todo_id), None)
                status = HTTPStatus.NO_CONTENT if removed else HTTPStatus.NOT_FOUND
                self.send_json(status)
            case _:
                self.send_error(HTTPStatus.NOT_FOUND)

`HTTPStatus` is an `IntEnum`, so it can be passed wherever a status code is expected — like `StatusCodes.Status201Created` / `HttpStatusCode.Created` in .NET:

In [ ]:
print(HTTPStatus.CREATED == 201, HTTPStatus.CREATED.phrase, HTTPStatus(404).name)

## `HTTPServer` vs `ThreadingHTTPServer`

- `HTTPServer` handles **one request at a time** — a slow request blocks everyone else.
- `ThreadingHTTPServer` handles each request on a **new thread** (with `daemon_threads = True`), like `HttpListener` with a thread per context. It's what `python -m http.server` uses.

Use `ThreadingHTTPServer` unless you have a reason not to — and then protect shared state with a lock (as `store_lock` does).

In [ ]:
server = ThreadingHTTPServer(("127.0.0.1", 0), TodoHandler)
BASE = f"http://127.0.0.1:{server.server_address[1]}"
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
print("todo API running:", server_thread.is_alive())

## Calling the API

A tiny client helper built on `urllib.request` that returns `(status, headers, body)` — and turns `HTTPError` into a normal return value so we can see 4xx responses:

In [ ]:
from urllib.request import Request
from urllib.error import HTTPError

def call(method, path, payload=None):
    data = None if payload is None else json.dumps(payload).encode()
    req = Request(BASE + path, data=data, method=method,
                  headers={"Content-Type": "application/json"} if data else {})
    try:
        with urlopen(req, timeout=5) as r:
            raw = r.read()
            return r.status, r.headers, json.loads(raw) if raw else None
    except HTTPError as e:
        raw = e.read()
        e.close()
        is_json = e.headers.get_content_type() == "application/json"
        return e.code, e.headers, json.loads(raw) if is_json else raw.decode().partition("\n")[0]

status, _, body = call("GET", "/todos")
print(status, body)

In [ ]:
status, headers, body = call("POST", "/todos", {"title": "review PR"})
print(status, headers["Location"], body)

status, _, body = call("GET", "/todos/2")
print(status, body)

status, _, body = call("POST", "/todos", {"wrong": "field"})
print(status, body)

In [ ]:
print(call("DELETE", "/todos/1")[0])        # 204 No Content
print(call("DELETE", "/todos/1")[0])        # 404 — already gone
print(call("GET", "/todos")[2])

Unhandled verbs and `send_error` produce standard responses automatically. A verb without a `do_<VERB>` method gets **501 Not Implemented**; `send_error` writes a small HTML error page:

In [ ]:
status, headers, body = call("GET", "/nope")
print(status, headers.get_content_type(), "|", body)

status, _, _ = call("PUT", "/todos/2", {"title": "x"})
print(status, HTTPStatus(status).phrase)

## What the handler sees: request details

Inside a handler you have:

| Attribute | Meaning | .NET (`HttpListenerRequest`) |
|---|---|---|
| `self.command` | `"GET"`, `"POST"`, ... | `HttpMethod` |
| `self.path` | path + query string, e.g. `/todos?done=1` | `RawUrl` |
| `self.headers` | case-insensitive `email.message.Message` | `Headers` |
| `self.rfile` | binary input stream | `InputStream` |
| `self.wfile` | binary output stream | `response.OutputStream` |
| `self.client_address` | `(host, port)` of the client | `RemoteEndPoint` |
| `self.request_version` | `"HTTP/1.1"` | `ProtocolVersion` |

⚠️ **Gotcha:** never call `self.rfile.read()` without a size — with keep-alive connections the stream doesn't end after the body, so the read **blocks forever**. Always read exactly `int(self.headers["Content-Length"])` bytes.

Stop the API server:

In [ ]:
server.shutdown()
server.server_close()
server_thread.join(timeout=2)
print("todo API stopped:", not server_thread.is_alive())

## Serving static files

`SimpleHTTPRequestHandler` serves files from a directory (with directory listings), like `UseStaticFiles()` + `UseDirectoryBrowser()`. Pass `directory=` with `functools.partial`, because the server constructs the handler itself:

In [ ]:
import functools
import tempfile
from pathlib import Path
from http.server import SimpleHTTPRequestHandler

class QuietFiles(SimpleHTTPRequestHandler):
    def log_message(self, format, *args):
        pass

with tempfile.TemporaryDirectory() as site:
    Path(site, "index.html").write_text("<h1>Product catalogue</h1>", encoding="utf-8")
    Path(site, "data.json").write_text('{"items": 3}', encoding="utf-8")

    handler = functools.partial(QuietFiles, directory=site)
    with ThreadingHTTPServer(("127.0.0.1", 0), handler) as static_server:
        port = static_server.server_address[1]
        t = threading.Thread(target=static_server.serve_forever, daemon=True)
        t.start()
        for name in ["index.html", "data.json"]:
            with urlopen(f"http://127.0.0.1:{port}/{name}", timeout=5) as r:
                print(name, "->", r.headers.get_content_type(), r.read().decode())
        static_server.shutdown()          # the with block then calls server_close()

## The `python -m http.server` command line

The same static file server is available from the shell — handy for quickly sharing a folder or previewing a static site, like `npx serve` or `dotnet serve`:

```bash
# serve the current directory on http://localhost:8000
python -m http.server

# choose port, bind address and directory
uv run python -m http.server 8080 --bind 127.0.0.1 --directory ./site
```

Press `Ctrl+C` to stop it. ⚠️ By default it listens on **all interfaces** — anyone on your network can read the folder. Use `--bind 127.0.0.1` for local-only access.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Server object | `HttpListener` / Kestrel | `ThreadingHTTPServer((host, port), Handler)` |
| Handle GET | `app.MapGet(...)` | `def do_GET(self):` |
| Route | route templates | `match` on `self.path` segments |
| Read body | `request.InputStream` | `self.rfile.read(int(self.headers["Content-Length"]))` |
| Status | `response.StatusCode` / `Results.*` | `self.send_response(HTTPStatus.OK)` |
| Header | `response.Headers.Add` | `self.send_header(k, v)` then `self.end_headers()` |
| Body | `response.OutputStream.Write` | `self.wfile.write(bytes)` |
| Error page | `Results.NotFound()` | `self.send_error(404)` |
| Run in background | `app.RunAsync()` | `Thread(target=server.serve_forever, daemon=True)` |
| Stop | `listener.Stop()` | `server.shutdown(); server.server_close()` |
| Static files | `UseStaticFiles()` | `SimpleHTTPRequestHandler` / `python -m http.server` |
| Production | ASP.NET Core + Kestrel | WSGI/ASGI framework + gunicorn/uvicorn |

## Exercises

1. Write a handler that answers `GET /add?a=2&b=3` with the JSON `{"result": 5.0}` and returns 400 with `{"error": ...}` if `a` or `b` is missing or not a number. Start it on port 0 in a background thread and call it with both a good and a bad query.
2. Add a `do_POST` to the same handler for `/upper` that reads a **plain-text** body and returns it upper-cased as `text/plain`.
3. Stop the server cleanly and confirm the serving thread has exited.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import json, threading
from http import HTTPStatus
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs
from urllib.request import urlopen, Request
from urllib.error import HTTPError

class CalcHandler(BaseHTTPRequestHandler):
    def reply(self, status, body, content_type="application/json"):
        data = body.encode() if isinstance(body, str) else json.dumps(body).encode()
        self.send_response(status)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    # 1 — query-string parameters with validation
    def do_GET(self):
        url = urlparse(self.path)
        if url.path != "/add":
            return self.send_error(HTTPStatus.NOT_FOUND)
        q = parse_qs(url.query)
        try:
            a, b = float(q["a"][0]), float(q["b"][0])
        except (KeyError, ValueError):
            return self.reply(HTTPStatus.BAD_REQUEST, {"error": "a and b must be numbers"})
        self.reply(HTTPStatus.OK, {"result": a + b})

    # 2 — plain-text body
    def do_POST(self):
        if self.path != "/upper":
            return self.send_error(HTTPStatus.NOT_FOUND)
        text = self.rfile.read(int(self.headers.get("Content-Length", 0))).decode()
        self.reply(HTTPStatus.OK, text.upper(), "text/plain; charset=utf-8")

    def log_message(self, *args):
        pass

srv = ThreadingHTTPServer(("127.0.0.1", 0), CalcHandler)
base = f"http://127.0.0.1:{srv.server_address[1]}"
thread = threading.Thread(target=srv.serve_forever, daemon=True)
thread.start()

with urlopen(f"{base}/add?a=2&b=3", timeout=5) as r:
    print(json.load(r))
try:
    urlopen(f"{base}/add?a=2", timeout=5)
except HTTPError as e:
    print(e.code, json.load(e))
    e.close()

req = Request(f"{base}/upper", data="hello server".encode(), method="POST",
              headers={"Content-Type": "text/plain"})
with urlopen(req, timeout=5) as r:
    print(r.read().decode())

# 3 — clean shutdown
srv.shutdown()
srv.server_close()
thread.join(timeout=2)
print("stopped:", not thread.is_alive())
```

</details>